# 01. Exploración de Datos (EDA)

En esta etapa, nos enfocaremos en entender la naturaleza del dataset de *California Housing*.

**Objetivo:** Obtener un entendimiento de los datos, de la industria, identificar anomalías, rangos, y relaciones clave.

### Instrucciones Generales:
1. **Carga los datos:** Lee el archivo `.csv` proveniente de la carpeta `data/raw/`
2. **Inspección:** Análisis exploratorio de datos, estructura, problemas de calidad: consistencia, sensibilidad, precisión y completitud.
3. **Visualización:** Genera gráficos que permitan entender la distribución de las variables y sus relaciones.

> **Nota:** Este notebook se enfoca en **analizar** y **documentar hallazgos**.  
> Las transformaciones definitivas del dataset se aplican en el Notebook 2.


In [ ]:
# Carga de librerías y dataset
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pandas.plotting import scatter_matrix

# Configuración visual básica
plt.rcParams['figure.figsize'] = (10, 6)
sns.set_style("whitegrid")

# Carga del dataset crudo
datos_crudos = pd.read_csv('../src/data/raw/housing/housing.csv')

# Vista inicial del DataFrame
print("Primeras 5 filas del dataset:")
display(datos_crudos.head())

print(f"Tipo de objeto: {type(datos_crudos)}")
print(f"Dimensiones del dataset: {datos_crudos.shape}")


## 1. Inspección general del dataset
Primero revisamos la estructura, los tipos de datos y la presencia inicial de valores faltantes.


In [ ]:
# Información general del dataset
datos_crudos.info()


In [ ]:
# Resumen estadístico de variables numéricas
datos_crudos.describe()


## 2. Completitud de datos

Analizamos valores nulos y su proporción para identificar problemas de completitud.


In [ ]:
# Resumen de valores nulos
resumen_nulos = pd.DataFrame({
    'valores_nulos': datos_crudos.isnull().sum(),
    'porcentaje_nulos': (datos_crudos.isnull().sum() / len(datos_crudos)) * 100
})

resumen_nulos['porcentaje_nulos'] = resumen_nulos['porcentaje_nulos'].round(4)
resumen_nulos = resumen_nulos.sort_values(by='porcentaje_nulos', ascending=False)
resumen_nulos_filtrado = resumen_nulos[resumen_nulos['valores_nulos'] > 0]

print("Resumen completo de valores nulos:")
display(resumen_nulos)

print("Columnas con valores nulos:")
display(resumen_nulos_filtrado)


In [ ]:
# Filas con valores nulos en total_bedrooms
filas_nulas_bedrooms = datos_crudos[datos_crudos['total_bedrooms'].isnull()]

print(f"Cantidad de filas con total_bedrooms nulo: {len(filas_nulas_bedrooms)}")
display(filas_nulas_bedrooms.head())


### Conclusión sobre completitud

La variable `total_bedrooms` presenta aproximadamente un 1% de valores faltantes.  
Dado que la proporción es baja, el problema de completitud es manejable y no compromete el análisis exploratorio.  
La decisión definitiva de imputación se aplica en el Notebook 2.


## 3. Rangos y revisión de sensibilidad / precisión

Aquí revisamos mínimos, máximos y posibles topes artificiales en variables relevantes.


In [ ]:
# Resumen de valores mínimos y máximos para variables numéricas
columnas_numericas = datos_crudos.select_dtypes(include=['float64', 'int64']).columns

resumen_rangos = pd.DataFrame({
    'valor_minimo': datos_crudos[columnas_numericas].min(),
    'valor_maximo': datos_crudos[columnas_numericas].max()
})

display(resumen_rangos)


In [ ]:
# Revisión de topes artificiales en edad y precio
registros_en_limite_edad = (datos_crudos['housing_median_age'] == 52).sum()
porcentaje_limite_edad = (registros_en_limite_edad / len(datos_crudos)) * 100

valor_max_precio = datos_crudos['median_house_value'].max()
registros_en_limite_precio = (datos_crudos['median_house_value'] == valor_max_precio).sum()
porcentaje_limite_precio = (registros_en_limite_precio / len(datos_crudos)) * 100

print(f"Registros con housing_median_age = 52: {registros_en_limite_edad} ({porcentaje_limite_edad:.2f}%)")
print(f"Registros con median_house_value = {valor_max_precio}: {registros_en_limite_precio} ({porcentaje_limite_precio:.2f}%)")


In [ ]:
# Histograma de edad y precio para revisar topes
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(datos_crudos['housing_median_age'], bins=50, kde=False, color='skyblue', ax=axes[0])
axes[0].set_title('Distribución de housing_median_age')

sns.histplot(datos_crudos['median_house_value'], bins=50, kde=False, color='salmon', ax=axes[1])
axes[1].set_title('Distribución de median_house_value')

plt.tight_layout()
plt.show()


## 4. Consistencia de datos

Validamos reglas lógicas y estructurales del dataset:
- columnas que conceptualmente deberían ser enteras,
- valores negativos no esperados,
- consistencia geográfica,
- consistencia categórica,
- y reglas lógicas entre habitaciones y dormitorios.


In [ ]:
# Columnas que conceptualmente deberían representar enteros
columnas_enteras_esperadas = [
    'housing_median_age',
    'total_rooms',
    'total_bedrooms',
    'population',
    'households'
]

inconsistencias_decimales = {}

for columna in columnas_enteras_esperadas:
    filas_inconsistentes = datos_crudos[
        datos_crudos[columna].notna() & (datos_crudos[columna] % 1 != 0)
    ]
    inconsistencias_decimales[columna] = filas_inconsistentes

for columna, tabla in inconsistencias_decimales.items():
    print(f"{columna}: {len(tabla)} filas con decimales no esperados")


In [ ]:
# Regla lógica: total_bedrooms no debería ser mayor que total_rooms
regla_rooms_bedrooms = datos_crudos[
    datos_crudos['total_bedrooms'].notna() &
    (datos_crudos['total_bedrooms'] > datos_crudos['total_rooms'])
]

print("Filas con total_bedrooms > total_rooms:", len(regla_rooms_bedrooms))
display(regla_rooms_bedrooms[[
    'longitude', 'latitude', 'total_rooms', 'total_bedrooms',
    'population', 'households', 'median_house_value', 'ocean_proximity'
]])


In [ ]:
# Revisión de valores negativos en variables de conteo
columnas_no_negativas = ['total_rooms', 'total_bedrooms', 'population', 'households']

for columna in columnas_no_negativas:
    negativos = datos_crudos[datos_crudos[columna].notna() & (datos_crudos[columna] < 0)]
    print(f"{columna}: {len(negativos)} filas con valores negativos")


In [ ]:
# Validación geográfica básica
latitudes_invalidas = datos_crudos[
    (datos_crudos['latitude'] < -90) | (datos_crudos['latitude'] > 90)
]

longitudes_invalidas = datos_crudos[
    (datos_crudos['longitude'] < -180) | (datos_crudos['longitude'] > 180)
]

print("Latitudes inválidas:", len(latitudes_invalidas))
print("Longitudes inválidas:", len(longitudes_invalidas))


In [ ]:
# Consistencia de categorías en ocean_proximity
print("Categorías únicas originales:")
display(pd.Series(sorted(datos_crudos['ocean_proximity'].dropna().unique()), name='ocean_proximity'))

print("Frecuencia por categoría:")
display(datos_crudos['ocean_proximity'].value_counts())

categorias_con_espacios = datos_crudos[
    datos_crudos['ocean_proximity'] != datos_crudos['ocean_proximity'].str.strip()
]

categorias_esperadas = {'<1H OCEAN', 'INLAND', 'ISLAND', 'NEAR BAY', 'NEAR OCEAN'}
categorias_reales = set(datos_crudos['ocean_proximity'].dropna().unique())
categorias_inesperadas = categorias_reales - categorias_esperadas
categorias_faltantes = categorias_esperadas - categorias_reales

print("Filas con espacios extra en ocean_proximity:", len(categorias_con_espacios))
print("Categorías inesperadas:", categorias_inesperadas)
print("Categorías faltantes:", categorias_faltantes)


### Conclusión sobre consistencia

Se detectaron 5 registros donde `total_bedrooms > total_rooms`, lo cual constituye una inconsistencia lógica.  
Dado que representan una proporción mínima del dataset, estos casos se eliminarán en el Notebook 2.  
El resto de validaciones permite concluir que el dataset mantiene una consistencia general aceptable para continuar con el análisis.


## 5. Histogramas de variables numéricas originales

Se generan histogramas individuales para las variables numéricas originales del dataset.


In [ ]:
# Histogramas individuales de las variables originales
columnas_originales = [
    'longitude',
    'latitude',
    'housing_median_age',
    'total_rooms',
    'total_bedrooms',
    'population',
    'households',
    'median_income',
    'median_house_value'
]

paleta_colores = [
    '#4C78A8', '#F58518', '#54A24B',
    '#E45756', '#72B7B2', '#B279A2',
    '#FF9DA6', '#9D755D', '#BAB0AC'
]

for columna, color in zip(columnas_originales, paleta_colores):
    plt.figure(figsize=(11, 5))
    plt.gcf().patch.set_facecolor('#f8f9fa')
    plt.gca().set_facecolor('#ffffff')

    plt.hist(
        datos_crudos[columna].dropna(),
        bins=30,
        color=color,
        edgecolor='black',
        linewidth=1.0,
        alpha=0.9
    )

    plt.title(f'Distribución de {columna}', fontsize=16, fontweight='bold', color='#333333')
    plt.xlabel(columna, fontsize=12)
    plt.ylabel('Frecuencia', fontsize=12)
    plt.grid(axis='y', linestyle=':', alpha=0.35)
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)
    plt.tight_layout()
    plt.show()


## 6. Visualización geoespacial

Generamos un gráfico de dispersión con `longitude` y `latitude`, coloreando los puntos según `median_house_value`.


In [ ]:
plt.figure(figsize=(10, 7))

scatter = plt.scatter(
    datos_crudos['longitude'],
    datos_crudos['latitude'],
    alpha=0.2,
    c=datos_crudos['median_house_value'],
    cmap='viridis',
    s=20
)

plt.title('Mapa geoespacial del valor medio de la vivienda', fontsize=16, fontweight='bold')
plt.xlabel('Longitud', fontsize=12)
plt.ylabel('Latitud', fontsize=12)

cbar = plt.colorbar(scatter)
cbar.set_label('Median House Value', fontsize=11)

plt.grid(alpha=0.2, linestyle='--')
plt.tight_layout()
plt.show()


## 7. Correlaciones y gráficos de dispersión

Calculamos la matriz de correlación frente a `median_house_value` y visualizamos relaciones relevantes.


In [ ]:
# Correlaciones de variables numéricas frente a median_house_value
matriz_correlacion = datos_crudos.corr(numeric_only=True)
correlacion_objetivo = matriz_correlacion['median_house_value'].sort_values(ascending=False)

display(correlacion_objetivo)


In [ ]:
# Scatter plots de variables numéricas relevantes frente a la variable objetivo
variables_clave = ['median_income', 'total_rooms', 'housing_median_age', 'latitude']

for variable in variables_clave:
    plt.figure(figsize=(8, 5))
    plt.scatter(
        datos_crudos[variable],
        datos_crudos['median_house_value'],
        alpha=0.2,
        c=datos_crudos['median_house_value'],
        cmap='viridis'
    )
    plt.title(f'{variable} vs median_house_value', fontsize=14, fontweight='bold')
    plt.xlabel(variable)
    plt.ylabel('median_house_value')
    plt.colorbar(label='Valor vivienda')
    plt.grid(alpha=0.2)
    plt.tight_layout()
    plt.show()


In [ ]:
# Matriz de dispersión para variables seleccionadas
atributos = ['median_house_value', 'median_income', 'total_rooms', 'housing_median_age']
scatter_matrix(datos_crudos[atributos], figsize=(12, 8), diagonal='hist')
plt.suptitle('Scatter Matrix de variables seleccionadas', y=1.02, fontsize=14, fontweight='bold')
plt.show()


## 8. Conclusiones del EDA

A partir del análisis exploratorio se identificaron los siguientes hallazgos clave:

- Existe un problema de completitud en `total_bedrooms`, aunque su magnitud es baja.
- Se detectaron 5 registros inconsistentes donde `total_bedrooms > total_rooms`.
- Hay evidencia de topes artificiales en `housing_median_age` y `median_house_value`.
- `ocean_proximity` requiere codificación para modelado.
- `median_income` aparece como una de las variables más prometedoras frente a `median_house_value`.

Estas decisiones se aplicarán formalmente en el Notebook 2 para construir el dataset limpio y enriquecido.
